In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import random
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import math
from accelerate import Accelerator

# diffusers imports
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL

import matplotlib.pyplot as plt

def denorm(x):
    """[-1, 1] -> [0, 1]"""
    return (x.clamp(-1, 1) + 1) / 2


def plot_triplet(img_prev, img_mid, img_next, idx=0):
    """
    img_prev, img_mid, img_next: [B, 3, H, W]
    idx: which sample in the batch to visualize
    """
    imgs = [img_prev, img_mid, img_next]
    titles = ["prev", "mid (gt)", "next"]

    plt.figure(figsize=(12, 4))
    for i, (img, title) in enumerate(zip(imgs, titles)):
        x = img[idx].detach().cpu()      # [3, H, W]
        x = denorm(x)
        x = x.permute(1, 2, 0)           # -> [H, W, 3]

        plt.subplot(1, 3, i + 1)
        plt.imshow(x)
        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

# -------------------------
# Utilities / small modules
# -------------------------
class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=768, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens



# -------------------------
# Dataset: neighbouring slices
# -------------------------

import os
import re
import random
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torchvision.transforms.functional as TF
from safetensors.torch import save_file, load_file


class Slice3DDataset(Dataset):
    """
    Filename format: <z>.png

    - z : slice index along z-axis (0, 1, ..., N)

    For the whole stack:
        choose endpoints (z_i, z_j) with:
            2 <= (z_j - z_i) <= max_gap

        for every z_t where i < t < j:
            input  : (z_i, z_j)
            target : z_t
            weights: (w_prev, w_next), sum to 1
    """

    def __init__(self, root_dir, max_gap=5):
        self.root_dir = root_dir
        self.max_gap = max_gap

        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                [0.5, 0.5, 0.5],
                [0.5, 0.5, 0.5]
            )
        ])

        # ------------------------------------------
        # Parse filenames: <z>.png
        # ------------------------------------------
        pattern = re.compile(r"^(\d+)\.png$")

        slices = []  # (z, path)
        for fname in os.listdir(root_dir):
            m = pattern.match(fname)
            if m is None:
                continue
            z = int(m.group(1))
            slices.append((z, os.path.join(root_dir, fname)))

        if len(slices) < 3:
            raise RuntimeError("Need at least 3 slices to build samples.")

        # sort by z index
        slices.sort(key=lambda x: x[0])
        zs = [z for z, _ in slices]

        # ------------------------------------------
        # Build interpolation samples
        # ------------------------------------------
        self.base_samples = []
        # each: (prev_path, next_path, gt_path, w_prev, w_next)

        n = len(slices)

        for i in range(n - 2):
            z_i, path_i = slices[i]

            # j-i 至少 2，最多 max_gap
            for j in range(i + 2, min(i + self.max_gap + 1, n)):
                z_j, path_j = slices[j]

                denom = z_j - z_i
                if denom <= 0:
                    continue

                for t in range(i + 1, j):
                    z_t, path_t = slices[t]

                    d_prev = z_t - z_i
                    d_next = z_j - z_t

                    w_prev = d_next / (d_prev + d_next)
                    w_next = d_prev / (d_prev + d_next)

                    self.base_samples.append(
                        (
                            path_i,
                            path_j,
                            path_t,
                            float(w_prev),
                            float(w_next),
                        )
                    )

        if len(self.base_samples) == 0:
            raise RuntimeError("No valid interpolation samples constructed.")

        # ------------------------------------------
        # deterministic augmentations (UNCHANGED)
        # ------------------------------------------
        self.num_rotations = 4
        self.num_hflip = 2
        self.num_vflip = 2
        self.num_aug = self.num_rotations * self.num_hflip * self.num_vflip

    def __len__(self):
        return len(self.base_samples) * self.num_aug

    def __getitem__(self, idx):
        base_idx = idx // self.num_aug
        rem = idx % self.num_aug

        rot_k = rem // (self.num_hflip * self.num_vflip)
        rem %= (self.num_hflip * self.num_vflip)
        hflip = rem // self.num_vflip
        vflip = rem % self.num_vflip

        prev_path, next_path, gt_path, w_prev, w_next = \
            self.base_samples[base_idx]

        img_prev = Image.open(prev_path).convert("RGB")
        img_next = Image.open(next_path).convert("RGB")
        img_gt   = Image.open(gt_path).convert("RGB")

        if rot_k > 0:
            angle = 90 * rot_k
            img_prev = img_prev.rotate(angle)
            img_next = img_next.rotate(angle)
            img_gt   = img_gt.rotate(angle)

        if hflip:
            img_prev = img_prev.transpose(Image.FLIP_LEFT_RIGHT)
            img_next = img_next.transpose(Image.FLIP_LEFT_RIGHT)
            img_gt   = img_gt.transpose(Image.FLIP_LEFT_RIGHT)

        if vflip:
            img_prev = img_prev.transpose(Image.FLIP_TOP_BOTTOM)
            img_next = img_next.transpose(Image.FLIP_TOP_BOTTOM)
            img_gt   = img_gt.transpose(Image.FLIP_TOP_BOTTOM)

        return (
            self.to_tensor(img_prev),
            self.to_tensor(img_next),
            self.to_tensor(img_gt),
            torch.tensor(w_prev, dtype=torch.float32),
            torch.tensor(w_next, dtype=torch.float32),
        )

In [ ]:
# infer_slice_interp.py
import os
import torch
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from safetensors.torch import load_file
from torch.utils.data import DataLoader

# -------------------------------------------------
# Utilities
# -------------------------------------------------
def denormalize(x):
    # [-1,1] -> [0,1]
    return (x.clamp(-1, 1) + 1) / 2


def show_result(prev, next, pred, gt, idx=0):
    imgs = [prev, next, pred, gt]
    titles = ["prev", "next", "pred_mid", "gt_mid"]

    plt.figure(figsize=(16, 4))
    for i, (img, title) in enumerate(zip(imgs, titles)):
        x = img[idx].detach().cpu()
        x = x.permute(1, 2, 0)
        x = denormalize(x)

        plt.subplot(1, 4, i + 1)
        plt.imshow(x)
        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# -------------------------------------------------
# Inference class
# -------------------------------------------------
class Slice3DInferencer:
    def __init__(
        self,
        unet_ckpt,
        cond_ckpt,
        pretrained_path="runwayml/stable-diffusion-v1-5",
        batch_size=4,
        device="cuda",
        num_tokens=64,
        cond_dim=768,
        num_inference_steps=200,
    ):
        self.device = torch.device(device)


        # -------------------------
        # Load VAE
        # -------------------------
        self.vae = AutoencoderKL.from_pretrained(
            pretrained_path, subfolder="vae"
        ).to(self.device)
        self.vae.eval()

        self.scaling_factor = getattr(self.vae.config, "scaling_factor", 0.18215)

        # -------------------------
        # Load UNet
        # -------------------------
        self.unet = UNet2DConditionModel.from_pretrained(
            pretrained_path, subfolder="unet"
        ).to(self.device)

        state = load_file(unet_ckpt)
        self.unet.load_state_dict(state, strict=True)
        self.unet.eval()

        # -------------------------
        # Load CondEncoder
        # -------------------------
        self.cond_proj = CondEncoder().to(self.device)
        state = load_file(cond_ckpt)
        self.cond_proj.load_state_dict(state, strict=True)
        self.cond_proj.eval()

        # -------------------------
        # Scheduler
        # -------------------------
        self.scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler"
        )
        self.scheduler.config.prediction_type = "sample"
        self.num_inference_steps = num_inference_steps

    @torch.no_grad()
    def infer_batch(self, batch):
        """
        batch: (img_prev, img_next, img_mid_gt)
        strength: how much noise to add to prev (img2img strength)
        """
        img_prev, img_next, img_mid_gt = batch
        img_prev = img_prev.to(self.device)
        img_next = img_next.to(self.device)
        img_mid_gt = img_mid_gt.to(self.device)

        self.vae.eval()
        self.unet.eval()
        self.cond_proj.eval()

        # --------------------------------------------------
        # 1. Encode prev / next
        # --------------------------------------------------
        latent_prev = self.vae.encode(img_prev).latent_dist.sample()
        latent_next = self.vae.encode(img_next).latent_dist.sample()

        latent_prev = latent_prev * self.scaling_factor
        latent_next = latent_next * self.scaling_factor

        # --------------------------------------------------
        # 2. Build condition (USE NEXT)
        # --------------------------------------------------
        condition = self.cond_proj(0.5*(latent_prev+latent_next))
        # shape: [B, num_tokens, cond_dim]

        # --------------------------------------------------
        # 3. Setup scheduler & timesteps
        # --------------------------------------------------
        self.scheduler.set_timesteps(self.num_inference_steps, device=self.device)
        timesteps = self.scheduler.timesteps

        # --------------------------------------------------
        # 4. Add noise to prev latent (img2img start)
        # --------------------------------------------------
        latents = torch.randn_like(latent_prev)

        # --------------------------------------------------
        # 5. Reverse diffusion
        # --------------------------------------------------
        for t in tqdm(timesteps, leave=False):
            # UNet predicts x0 (latent_mid)
            x0_pred = self.unet(
                latents,
                t,
                encoder_hidden_states=condition
            ).sample

            latents = self.scheduler.step(
                model_output=x0_pred,
                timestep=t,
                sample=latents
            ).prev_sample

        # --------------------------------------------------
        # 6. Decode
        # --------------------------------------------------
        latents = latents / self.scaling_factor
        img_mid_pred = self.vae.decode(latents).sample

        return img_prev, img_next, img_mid_pred, img_mid_gt

    def run(self, num_batches=5, visualize=True):
        for i, batch in enumerate(self.val_loader):
            if i >= num_batches:
                break

            prev, next_, pred, gt = self.infer_batch(batch)

            if visualize:
                show_result(prev, next_, pred, gt, idx=0)

    @torch.no_grad()
    def infer(self, img_prev, img_next, wp, wn, name):
        """
        batch: (img_prev, img_next, img_mid_gt)
        strength: how much noise to add to prev (img2img strength)
        """
        img_prev = img_prev.to(self.device)
        img_next = img_next.to(self.device)

        self.vae.eval()
        self.unet.eval()
        self.cond_proj.eval()

        # --------------------------------------------------
        # 1. Encode prev / next
        # --------------------------------------------------
        latent_prev = self.vae.encode(img_prev).latent_dist.sample()
        latent_next = self.vae.encode(img_next).latent_dist.sample()

        latent_prev = latent_prev * self.scaling_factor
        latent_next = latent_next * self.scaling_factor

        # --------------------------------------------------
        # 2. Build condition (USE NEXT)
        # --------------------------------------------------
        condition = self.cond_proj(wp*latent_prev + wn*latent_next)
        # shape: [B, num_tokens, cond_dim]

        # --------------------------------------------------
        # 3. Setup scheduler & timesteps
        # --------------------------------------------------
        self.scheduler.set_timesteps(self.num_inference_steps, device=self.device)
        timesteps = self.scheduler.timesteps

        # --------------------------------------------------
        # 4. Add noise to prev latent (img2img start)
        # --------------------------------------------------
        latents = torch.randn_like(latent_prev)

        # --------------------------------------------------
        # 5. Reverse diffusion
        # --------------------------------------------------
        for t in tqdm(timesteps, leave=False):
            # UNet predicts x0 (latent_mid)
            x0_pred = self.unet(
                latents,
                t,
                encoder_hidden_states=condition
            ).sample

            latents = self.scheduler.step(
                model_output=x0_pred,
                timestep=t,
                sample=latents
            ).prev_sample

        # --------------------------------------------------
        # 6. Decode
        # --------------------------------------------------
        latents = latents / self.scaling_factor
        torch.save(latents, name)
        img_mid_pred = self.vae.decode(latents).sample

        return img_prev, img_next, img_mid_pred


In [ ]:
infer = Slice3DInferencer(
    unet_ckpt="drive/MyDrive/checkpoints_3dfudan/unet.safetensors",
    cond_ckpt="drive/MyDrive/checkpoints_3dfudan/condencoder.safetensors",
    device="cuda"
)


In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

infer.num_inference_steps = 200

# ====== load & preprocess images ======
img_prev = Image.open("65.png").convert("RGB")
img_next = Image.open("66.png").convert("RGB")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

img_prev = transform(img_prev)
img_next = transform(img_next)

# ====== output dir ======
out_dir = "./outputs"
os.makedirs(out_dir, exist_ok=True)

# ====== weight sweep ======
weights = [i / 10 for i in range(1, 10)]  # 0.1 ... 0.9

for w_prev in weights:
    w_next = 1.0 - w_prev

    w_prev_t = torch.tensor(w_prev)
    w_next_t = torch.tensor(w_next)

    name = f"{w_prev:.1f}_{w_next:.1f}.pt"

    img_prev_o, img_next_o, img_mid_pred = infer.infer(
        img_prev.unsqueeze(0),
        img_next.unsqueeze(0),
        w_prev_t,
        w_next_t,
        name
    )

    show_result(img_prev_o, img_next_o, img_mid_pred, img_mid_pred)

    # ====== save PNG ======
    x = img_mid_pred.detach().cpu()
    x = denormalize(x)
    x = x[0].permute(1, 2, 0)

    if x.max() <= 1.0:
        x_uint8 = (x * 255.0).round().clamp(0, 255).to(torch.uint8)
    else:
        x_uint8 = x.round().clamp(0, 255).to(torch.uint8)

    x_np = x_uint8.numpy()

    png_name = f"mid_{w_prev:.3f}_{w_next:.3f}.png"
    Image.fromarray(x_np).save(
        os.path.join(out_dir, png_name),
        format="PNG"
    )

    print(f"Saved: {name}, {png_name}")
